# NGRAM = TRUE
# MIN_COUNT = 10
# EMBEDDER = all-MiniLM-L6-v2

In [1]:
!pip install gensim top2vec


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from top2vec import Top2Vec
import pandas as pd
from bertopic import BERTopic
from bertopic.backend import MultiModalBackend
from bertopic.representation import KeyBERTInspired
import re
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import random
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

c:\Users\Allen\Documents\Python Env\environments\derp_learning\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv(r"C:\Users\Allen\Documents\GitHub\thesis-research\combined_tweets_dataset\sample_v2\vibe_coding_combined_translated.csv")

In [4]:
df = df[["full_text_translated", "image_url"]]

In [5]:
df.head()

,full_text_translated,image_url
0,@karpathy me vibe coding and hope it can just ...,https://pbs.twimg.com/media/Gi0a82HaQAA97Q-.jpg
1,vibes capital meets vibe coding = greatest eco...,NaN
2,vibe coding https://t.co/1hHVMmunrw,https://pbs.twimg.com/media/Gi0f9fAWgAAgox6.jpg
3,can attest. vibe coding is hella fun and borde...,NaN
4,I have built a few app ideas in my spare time ...,NaN


In [6]:
def preprocess_tweet(text):
    #lower case text
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove user mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags
    text = re.sub(r'#\w+', '', text)
    #remove symbols
    text = re.sub(r'[^\w\s]', '', text)
    #remove numbers
    text = re.sub(r'\d+', '', text)
    #Replace all "vibe code" into vibecode
    text = re.sub(r'vibe code', 'vibecode', text)
    #Replace all "vibe coding" into vibecoding
    text = re.sub(r'vibe coding', 'vibecoding', text)
    #Relace all "vibe coded" into vibecoded
    text = re.sub(r'vibe coded', 'vibecoded', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [7]:
df_preprocessed = df['full_text_translated'].fillna('').apply(preprocess_tweet)
df_preprocessed = pd.concat([df_preprocessed, df['image_url']], axis=1)

In [8]:
df_preprocessed = df_preprocessed[df_preprocessed['full_text_translated'].apply(lambda x: len(x.split()) >= 4)]
df_preprocessed.count()

full_text_translated    20511
image_url                6495
dtype: int64

In [9]:
docs = df_preprocessed['full_text_translated'].tolist()
images  = df_preprocessed['image_url'].tolist()
for i in range(len(images)):
    if pd.isna(images[i]):
        images[i] = None
random.shuffle(docs)
docs[:10]

['kalshi taking k out of peoples accounts because they cant vibecode their rewards system right lmaooo',
 'anyone heard of this claude thing i think it could be huge you just finance a mac mini and start vibecoding were probably gonna change the world with this one imo if you havent tried it you definitely should but tbh if u arent already using claude its probably over for you',
 'its a lot more expensive to vibecode software or hire software engineers who can use ai the state of outsourcing software engineering to ai or vibecoders is about the same as the state of outsourcing medical treatment to ai and yes many idiots do believe that ai gt mds',
 'also why everyone started vibecoding during last weeks what happened exactly can someone tell the news or maybe therre is something in the water now',
 'so much vibecoding and so little vibe pming',
 'vibecoding art i made with cursor and photoshop used cursor to make python script that detects contours and generates boxes around them left

In [10]:
# embedding_model = SentenceTransformer("all-mpnet-base-v2")

umap_model = UMAP(n_neighbors=10, n_components=5, min_dist=0.0)
hdbscan_model = HDBSCAN(min_cluster_size=15, min_samples=1)
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_df=0.8,
    min_df=2
)

representation_model = KeyBERTInspired()

In [11]:
# Top2Vec topic modeling
documents = [doc for doc in docs if isinstance(doc, str) and doc.strip()]
tokenized_documents = [doc.split() for doc in documents]

dictionary = Dictionary(tokenized_documents)
dictionary.filter_extremes(no_below=2, no_above=0.5)

print(f"tokenized documents: {tokenized_documents[:5]}")
print(f"Number of documents: {len(documents)}")
print(f"Number of unique tokens: {len(dictionary)}")



tokenized documents: [['kalshi', 'taking', 'k', 'out', 'of', 'peoples', 'accounts', 'because', 'they', 'cant', 'vibecode', 'their', 'rewards', 'system', 'right', 'lmaooo'], ['anyone', 'heard', 'of', 'this', 'claude', 'thing', 'i', 'think', 'it', 'could', 'be', 'huge', 'you', 'just', 'finance', 'a', 'mac', 'mini', 'and', 'start', 'vibecoding', 'were', 'probably', 'gonna', 'change', 'the', 'world', 'with', 'this', 'one', 'imo', 'if', 'you', 'havent', 'tried', 'it', 'you', 'definitely', 'should', 'but', 'tbh', 'if', 'u', 'arent', 'already', 'using', 'claude', 'its', 'probably', 'over', 'for', 'you'], ['its', 'a', 'lot', 'more', 'expensive', 'to', 'vibecode', 'software', 'or', 'hire', 'software', 'engineers', 'who', 'can', 'use', 'ai', 'the', 'state', 'of', 'outsourcing', 'software', 'engineering', 'to', 'ai', 'or', 'vibecoders', 'is', 'about', 'the', 'same', 'as', 'the', 'state', 'of', 'outsourcing', 'medical', 'treatment', 'to', 'ai', 'and', 'yes', 'many', 'idiots', 'do', 'believe', 'tha

In [12]:
print(dictionary.token2id)

{'accounts': 0, 'because': 1, 'cant': 2, 'k': 3, 'kalshi': 4, 'lmaooo': 5, 'of': 6, 'out': 7, 'peoples': 8, 'rewards': 9, 'right': 10, 'system': 11, 'taking': 12, 'their': 13, 'they': 14, 'vibecode': 15, 'a': 16, 'already': 17, 'and': 18, 'anyone': 19, 'arent': 20, 'be': 21, 'but': 22, 'change': 23, 'claude': 24, 'could': 25, 'definitely': 26, 'finance': 27, 'for': 28, 'gonna': 29, 'havent': 30, 'heard': 31, 'huge': 32, 'i': 33, 'if': 34, 'imo': 35, 'it': 36, 'its': 37, 'just': 38, 'mac': 39, 'mini': 40, 'one': 41, 'over': 42, 'probably': 43, 'should': 44, 'start': 45, 'tbh': 46, 'thing': 47, 'think': 48, 'this': 49, 'tried': 50, 'u': 51, 'using': 52, 'were': 53, 'with': 54, 'world': 55, 'you': 56, 'about': 57, 'ai': 58, 'as': 59, 'believe': 60, 'can': 61, 'do': 62, 'engineering': 63, 'engineers': 64, 'expensive': 65, 'gt': 66, 'hire': 67, 'idiots': 68, 'is': 69, 'lot': 70, 'many': 71, 'medical': 72, 'more': 73, 'or': 74, 'outsourcing': 75, 'same': 76, 'software': 77, 'state': 78, 'tha

In [13]:
import top2vec
print(top2vec.__version__)

1.0.36


In [14]:
top2vec_model = Top2Vec(
    documents=documents,
    ngram_vocab=True,
    min_count=10,
    embedding_model='all-MiniLM-L6-v2',    
    speed='learn',
    workers=2,
    keep_documents=True,
    verbose=True,
 )

topic_words, topic_word_scores, topic_nums = top2vec_model.get_topics()



2026-04-12 15:01:26,379 - top2vec - INFO - Pre-processing documents for training
2026-04-12 15:01:28,034 - top2vec - INFO - Downloading all-MiniLM-L6-v2 model
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6137.51it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-04-12 15:01:33,402 - top2vec - INFO - Creating joint document/word embedding
2026-04-12 15:01:40,773 - top2vec - INFO - Creating lower dimension embedding of documents
2026-04-12 15:01:59,176 - top2vec - INFO - Finding dense areas of documents
2026-04-12 15:02:00,631 - top2vec - INFO - Finding topics


In [15]:
topic_sizes, topic_ids = top2vec_model.get_topic_sizes()
pd.DataFrame({
    "topic_id": topic_ids,
    "size": topic_sizes
}).sort_values("size", ascending=False).head(10)

,topic_id,size
0,0,20036
1,1,256
2,2,219


In [16]:
print(len(topic_sizes))

3


In [18]:
# Coherence metrics for Top2Vec topics (C_V and NPMI)
topic_pairs = []
for topic_num, topic in zip(topic_nums, topic_words):
    cleaned_topic_terms = []
    for term in topic[:10]:
        # ngram terms like "vibe coding" are split so they can be matched to dictionary tokens
        for tok in str(term).split():
            if tok in dictionary.token2id:
                cleaned_topic_terms.append(tok)
    # keep token order, drop duplicates
    cleaned_topic_terms = list(dict.fromkeys(cleaned_topic_terms))
    if len(cleaned_topic_terms) >= 2:
        topic_pairs.append((topic_num, cleaned_topic_terms))

topics_for_coherence = [words for _, words in topic_pairs]
topic_nums_for_coherence = [topic_num for topic_num, _ in topic_pairs]

if not topics_for_coherence:
    raise ValueError("No valid topics left for coherence calculation. Try lowering dictionary filtering or increasing topic words.")

coherence_cv_model = CoherenceModel(
    topics=topics_for_coherence,
    texts=tokenized_documents,
    dictionary=dictionary,
    coherence='c_v',
)

coherence_npmi_model = CoherenceModel(
    topics=topics_for_coherence,
    texts=tokenized_documents,
    dictionary=dictionary,
    coherence='c_npmi',
)

coherence_cv = coherence_cv_model.get_coherence()
coherence_npmi = coherence_npmi_model.get_coherence()

print(f"Total topics used for coherence: {len(topics_for_coherence)}")
print(f"C_V coherence: {coherence_cv:.4f}")
print(f"NPMI coherence: {coherence_npmi:.4f}")

topic_coherence = pd.DataFrame({
    'topic_num': topic_nums_for_coherence,
    'topic_words': [' '.join(words) for words in topics_for_coherence],
    'c_v': coherence_cv_model.get_coherence_per_topic(),
    'npmi': coherence_npmi_model.get_coherence_per_topic(),
})

topic_coherence.sort_values('c_v', ascending=False).head(10)

Total topics used for coherence: 3
C_V coherence: 0.4127
NPMI coherence: -0.2463


,topic_num,topic_words,c_v,npmi
1,1,dance clubfor vibecode camp vibe designing vib...,0.488107,-0.303005
2,2,rust workshop vibecoded projects vibecode camp...,0.393847,-0.246998
0,0,vibecoded projects vibe engineering designing ...,0.356129,-0.188880


In [24]:
topic_words_values = topic_coherence.sort_values('c_v', ascending=False).head(10)['topic_words'].values
print(topic_words_values)

['dance clubfor vibecode camp vibe designing vibecoded projects dreamy weirdo indie hackers punk rock living agentforce vibes investing'
 'rust workshop vibecoded projects vibecode camp vibe engineering debugging designing trading cleanup ltcodinggt investing'
 'vibecoded projects vibe engineering designing marketing debugging vibecode camp investing trading ltcodinggt cleanup']


In [23]:
# Save model
top2vec_model.save("top2vec_ngram_c10.model")

# Load model (di masa depan)
# from top2vec import Top2Vec
top2vec_model = Top2Vec.load("top2vec_ngram_c10.model")



# Load di kemudian hari
# embeddings = np.load("top2vec_ngram_c10_embeddings.npy")


# Simpan Dictionary
dictionary.save("top2vec_ngram_c10.dict")

# Simpan tokenized_documents (menggunakan pickle karena list of lists)
import pickle
with open("top2vec_ngram_c10_tokenized_docs.pkl", "wb") as f:
    pickle.dump(tokenized_documents, f)
    
    
    
# Simpan ringkasan topik dan metriknya
topic_coherence.to_csv("top2vec_ngram_c10_metrics_summary.csv", index=False)

# Simpan representasi topik (10 kata teratas)
top_words_df = pd.DataFrame(topic_words)
top_words_df.to_csv("top2vec_ngram_c10_topic_words.csv")